In [1]:
import requests
import pandas as pd
import os

LABELS = ["WALKING", "STANDING", "SITTING", "LAYING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS"]

def call_prediction_api(window_vals, label_counts):
    url = "http://127.0.0.1:8006/predict"
    data = {"samples": window_vals.to_dict(orient="records")}
    print(data)
    response = requests.post(url, json=data)
    if response.status_code == 200:
        print("API Response:", response.json())
        predicted = response.json()["activity"]

        row = {label: label_counts.get(label, 0) for label in LABELS}
        row["predicted"] = predicted

        csv_path = "prediction_report.csv"
        file_exists = os.path.exists(csv_path)
        pd.DataFrame([row]).to_csv(csv_path, mode="a", header=not file_exists, index=False)

    else:
        print("API Error:", response.status_code, response.text)


In [2]:

import pandas as pd
import time

data_path = r"/home/pitoleo/src/neat-calculator/neat_dashboard/merged_labeled_data_noisy.csv"

df = pd.read_csv(data_path)

# Keep the label column to save with windows
# df = df.drop(columns=['label'])


window = 128
step = 64
index = 0

X_windows = []


for start in range(0, len(df) - window + 1, step):
    end = start + window
    window_vals = df[start:end].copy()
    
    # Replace timestamps with current time + 20ms increments
    current_time_ms = int(time.time() * 1000)
    for i in range(len(window_vals)):
        window_vals.iloc[i, window_vals.columns.get_loc('timestamp')] = current_time_ms + (i * 20)
    
    # Add window_id to track which window this belongs to
    # window_vals['window_id'] = index
    
    X_windows.append(window_vals)

    # print(X_windows[0])
    label_counts = window_vals["label"].value_counts()
    print(window_vals["label"].value_counts().to_string())
    call_prediction_api(window_vals, label_counts)

    time.sleep(0.02)
    
    index += 1
    

# print(f"Created {len(X_windows)} windows of size {window} with step {step} out of {len(df)} data points.")

# # Concatenate all windows into a single DataFrame
# all_windows_df = pd.concat(X_windows, ignore_index=True)

# # Group by window_id and get the most prevalent label for each window
# if 'label' in all_windows_df.columns:
#     window_labels = all_windows_df.groupby('window_id')['label'].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0])
#     window_summary = pd.DataFrame({
#         'window_id': window_labels.index,
#         'label': window_labels.values
#     })
# else:
#     window_summary = pd.DataFrame({
#         'window_id': all_windows_df['window_id'].unique()
#     })

# window_summary.to_csv(r"E:\src\neat-calculator\neat_dashboard\windows.csv", index=False)

# print(f"Saved {len(window_summary)} windows with their labels to windows.csv")


label
WALKING_DOWNSTAIRS    128
{'samples': [{'accelerometerX': -1.5411010438654509, 'accelerometerY': -10.115290403933018, 'accelerometerZ': -0.8997517736050178, 'gyroscopeX': -0.2710612903522166, 'gyroscopeY': -0.4302242436966463, 'gyroscopeZ': 0.0068527423064786, 'timestamp': 1772529463448, 'timestampNanos': 1104894633860846, 'label': 'WALKING_DOWNSTAIRS'}, {'accelerometerX': -1.7420711691854336, 'accelerometerY': -9.792017133808947, 'accelerometerZ': -1.1109010803030352, 'gyroscopeX': -0.528106950912045, 'gyroscopeY': -0.1363768854854731, 'gyroscopeZ': -0.0021471418086843, 'timestamp': 1772529463468, 'timestampNanos': 1104894668029543, 'label': 'WALKING_DOWNSTAIRS'}, {'accelerometerX': -1.4259046888420277, 'accelerometerY': -9.965875484731532, 'accelerometerZ': -1.3549265215994373, 'gyroscopeX': -0.2881663342185667, 'gyroscopeY': -0.1442875246373714, 'gyroscopeZ': -0.0596812096012559, 'timestamp': 1772529463488, 'timestampNanos': 1104894688167616, 'label': 'WALKING_DOWNSTAIRS'}, {'

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8006): Max retries exceeded with url: /predict (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8006): Failed to establish a new connection: [Errno 111] Connection refused"))